# combRUM Quickstart

This notebook builds the smallest complete combRUM example: a small bundle-choice model, an oracle that solves the agent's choice problem, a feature map, estimation, and a bootstrap.

For each observation $i$ and simulation draw $s$, combRUM asks the oracle for the best bundle at the current parameter vector:

$$d^*_{is}(\theta) \in \arg\max_{d \in \mathcal{D}_i} \phi_i(d)^\top \theta + \varepsilon_{is}(d).$$

In this notebook the demand oracle is just solving the combinatorial maximization by brute-force.

In combRUM, you provide the model-specific pieces: how to solve the choice problem and how to compute features for a bundle. combRUM handles row generation and the LP used for estimation and bootstrap.

## Data

A small, well-conditioned bundle-choice panel: each item has three observed covariates, each agent chooses any subset of four items, and simulated shocks enter item utility additively.

In [1]:
import itertools

import numpy as np

import combrum as cb

rng = np.random.default_rng(17)

N_ITEMS = 4
N_FEATURES = 3
N_OBS = 100
N_SIMULATIONS = 4
THETA_TRUE = np.array([0.9, -0.5, 0.35])
SHOCK_SCALE = 0.10

BUNDLES = np.array(
    list(itertools.product([0.0, 1.0], repeat=N_ITEMS))
)
X = rng.normal(size=(N_OBS, N_ITEMS, N_FEATURES))
estimation_shocks = rng.normal(
    scale=SHOCK_SCALE,
    size=(N_OBS, N_SIMULATIONS, N_ITEMS),
)
observed_shocks = rng.normal(scale=SHOCK_SCALE, size=(N_OBS, N_ITEMS))


def simulate_observed(theta):
    observed = np.zeros((N_OBS, N_ITEMS))
    for i in range(N_OBS):
        utilities = X[i] @ theta + observed_shocks[i]
        scores = BUNDLES @ utilities
        observed[i] = BUNDLES[np.argmax(scores)]
    return observed


observed_bundles = simulate_observed(THETA_TRUE)

{
    "observed_bundles": observed_bundles.shape,
    "mean_bundle_size": observed_bundles.sum(axis=1).mean(),
    "choice_share_by_item": observed_bundles.mean(axis=0).round(3).tolist(),
}

{'observed_bundles': (100, 4),
 'mean_bundle_size': 2.04,
 'choice_share_by_item': [0.49, 0.57, 0.48, 0.5]}

## Oracle and feature map

`BundleOracle` solves the bundle-choice problem for a batch of observations. Here the choice set is small, so the oracle simply tries every feasible bundle and returns the best one.

`BundleFeatures` turns each chosen bundle into the feature row used in estimation. combRUM uses the same feature map to compute the features at observed bundles and for adding constraints iteratively during row generation.

In [2]:
class BundleOracle(cb.Oracle):
    def price_batch(self, theta, local_ids):
        i = local_ids % N_OBS
        s = local_ids // N_OBS
        utilities = X[i] @ theta + estimation_shocks[i, s]
        scores = utilities @ BUNDLES.T
        choices = np.argmax(scores, axis=1)
        payoffs = scores[np.arange(local_ids.size), choices]
        return cb.DemandBatch.exact(local_ids, BUNDLES[choices], payoffs)


class BundleFeatures(cb.FeatureMap):
    def features_batch(self, ids, bundles):
        i = ids % N_OBS
        s = ids // N_OBS
        return np.einsum("ij,ijk->ik", bundles, X[i]), np.einsum(
            "ij,ij->i", bundles, estimation_shocks[i, s]
        )

## Model and data

`Parameters` defines the coefficients to estimate and their bounds. `Model` combines the oracle, feature map, and parameter layout. `Data` contains the observed choices, simulation draws, and covariates used in the fit.

In [3]:
parameters = cb.Parameters({"taste": (-2.0, 2.0, N_FEATURES)})
features = BundleFeatures()
oracle = BundleOracle()
model = cb.Model(
    oracle,
    parameters,
    features=features,
    formulation=cb.NSlack,
)
data = cb.Data(
    observed_bundles=observed_bundles,
    shocks=estimation_shocks,
    observables=X,
)
theta_true = parameters.pack({"taste": THETA_TRUE})

{
    "parameters": repr(parameters),
    "blocks": parameters.names,
    "theta_true": parameters.unpack(theta_true),
    "formulation": model.formulation.__name__,
}

{'parameters': 'Parameters(K=3: taste[3])',
 'blocks': ('taste',),
 'theta_true': {'taste': array([ 0.9 , -0.5 ,  0.35])},
 'formulation': 'NSlack'}

## Point estimate

`estimate` runs row generation. We compare the estimated coefficients with the true values used to simulate the data.

In [4]:
fit = cb.estimate(
    model,
    data,
    master_backend="highs",
    tolerance=1e-8,
    max_iterations=100,
    return_slack=True,
)

recovery_error = fit.theta_hat - theta_true

{
    "theta_true": THETA_TRUE.round(4).tolist(),
    "theta_hat": fit.theta_named()["taste"].round(4).tolist(),
    "recovery_error": recovery_error.round(4).tolist(),
    "max_abs_recovery_error": round(np.max(np.abs(recovery_error)), 4),
    "objective": round(fit.objective, 4),
    "converged": fit.metadata["converged"],
    "iterations": fit.metadata["iterations"],
    "active_cuts": fit.n_active_cuts,
    "max_slack": round(fit.slack_summary()["max_slack"], 4),
}

{'theta_true': [0.9, -0.5, 0.35],
 'theta_hat': [0.8834, -0.4872, 0.3418],
 'recovery_error': [-0.0166, 0.0128, -0.0082],
 'max_abs_recovery_error': 0.0166,
 'objective': 4.0995,
 'converged': True,
 'iterations': 5,
 'active_cuts': 751,
 'max_slack': 4.8383}

## Bootstrap

`bootstrap` runs a Bayesian bootstrap to compute standard errors and confidence intervals for the estimated coefficients.

In [5]:
boot = cb.bootstrap(
    model,
    data,
    n_bootstrap=10,
    weights=cb.NativeDraws(n_obs=N_OBS, base_seed=23),
    master_backend="highs",
    tolerance=1e-8,
    max_iterations=100,
)

ci_lo, ci_hi = boot.ci(only_converged=False)
{
    "n_converged": boot.n_converged,
    "mean": boot.mean(only_converged=False).round(4).tolist(),
    "se": boot.se_named(only_converged=False)["taste"].round(4).tolist(),
    "ci_95": [ci_lo.round(4).tolist(), ci_hi.round(4).tolist()],
}

{'n_converged': 10,
 'mean': [0.992, -0.5423, 0.3912],
 'se': [0.1439, 0.0716, 0.0606],
 'ci_95': [[0.775, -0.6194, 0.2955], [1.2267, -0.3995, 0.4855]]}

## Warm-start a follow-up fit

A warm start reuses constraints from an earlier fit. Here we keep the same model and data, but ask for a tighter tolerance, so the earlier constraints can be reused.

In [6]:
cold_start_fit = cb.estimate(
    model,
    data,
    master_backend="highs",
    tolerance=1e-4,
    max_iterations=30,
    return_cuts=True,
)
warm_start_fit = cb.estimate(
    model,
    data,
    master_backend="highs",
    tolerance=1e-8,
    max_iterations=100,
    warm_start=cold_start_fit,
    warm_cuts=cold_start_fit.cuts,
)

{
    "cold_start_iterations": cold_start_fit.metadata["iterations"],
    "warm_start_iterations": warm_start_fit.metadata["iterations"],
    "warm_start_theta": warm_start_fit.theta_named()["taste"].round(4).tolist(),
    "same_solution": np.allclose(cold_start_fit.theta_hat, warm_start_fit.theta_hat),
}

{'coarse_iterations': 5,
 'polished_iterations': 1,
 'polished_theta': [0.8834, -0.4872, 0.3418],
 'same_solution': True}

## Next

- `notebooks/02_blp_bundle_demand.ipynb`: bundle demand with endogenous prices.
- `notebooks/03_network_formation.ipynb`: network formation with reciprocity.
- `notebooks/04_unitdemand_blp_large.ipynb`: BLP inversion with many agents per market.
- `notebooks/05_peer_effects_large_network.ipynb`: peer effects on a large network.
- `notebooks/06_combinatorial_auction.ipynb`: a combinatorial auction example.